# 02 False Branch vs True

A toy demo showing a key inverse-problem pattern:

- optimization can converge stably
- multiple branches can satisfy part of the constraint stack
- one branch is the intended or "true" branch
- another branch can remain stable but wrong until an extra orthogonal constraint is added


## Setup

We define a simple 2-parameter score function over `(x, y)` with:

- a circular observable
- a diagonal compatibility constraint
- a weak branch preference term that still leaves two attractive regions
- a final orthogonal constraint that selects the true branch

This is a toy model only. The point is structural:

> stable convergence does not guarantee the correct branch.


In [ ]:
# Colab setup (safe even if already installed)
!pip -q install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print('Constraint Branch Selection Demo')
print('stable convergence ≠ correct branch')

In [ ]:
# Parameter grid
x = np.linspace(-2.2, 2.2, 600)
y = np.linspace(-2.2, 2.2, 600)
X, Y = np.meshgrid(x, y)

# Base losses
# L1: circle of radius sqrt(2)
L1 = (X**2 + Y**2 - 2.0)**2

# L2: near the diagonal y = x
L2 = (Y - X)**2

# L3: weak tilt toward the lower-right branch
# this creates a stable-but-wrong preference if used too early
L3 = (Y + 0.85)**2 + 0.15*(X - 1.0)**2

# L4: orthogonal constraint favoring the upper-right branch
L4 = (Y - 1.0)**2 + 0.08*(X - 1.0)**2

# Combined losses
score_partial = 1.2*L1 + 0.8*L2 + 0.20*L3
score_full = 1.2*L1 + 0.8*L2 + 0.20*L3 + 1.4*L4

# Best points
ij_partial = np.unravel_index(np.argmin(score_partial), score_partial.shape)
ij_full = np.unravel_index(np.argmin(score_full), score_full.shape)

partial_best = (X[ij_partial], Y[ij_partial], score_partial[ij_partial])
full_best = (X[ij_full], Y[ij_full], score_full[ij_full])

print('Partial best (stable but potentially wrong branch):', partial_best)
print('Full best (after orthogonal constraint):', full_best)

## Plot 1 — Partial constraint stack

With only the partial stack, optimization is stable, but it prefers a branch that is not the intended final one.


In [ ]:
plt.figure(figsize=(7, 7))
levels = np.linspace(np.min(score_partial), np.percentile(score_partial, 8), 18)
plt.contour(X, Y, score_partial, levels=levels)
plt.scatter([partial_best[0]], [partial_best[1]], marker='x', s=120)
plt.text(partial_best[0] + 0.06, partial_best[1] - 0.08, 'stable wrong branch')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Partial stack: stable convergence can select the wrong branch')
plt.axis('equal')
plt.show()

## Plot 2 — Full constraint stack

After adding an orthogonal constraint, the preferred solution shifts to the intended branch.


In [ ]:
plt.figure(figsize=(7, 7))
levels = np.linspace(np.min(score_full), np.percentile(score_full, 8), 18)
plt.contour(X, Y, score_full, levels=levels)
plt.scatter([full_best[0]], [full_best[1]], marker='x', s=120)
plt.text(full_best[0] + 0.06, full_best[1] + 0.06, 'true branch selected')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Full stack: orthogonal constraint selects the true branch')
plt.axis('equal')
plt.show()

## Plot 3 — Side-by-side minima locations

This plot overlays the two selected points on the same geometric background.


In [ ]:
circle_mask = np.abs(X**2 + Y**2 - 2.0) < 0.05

plt.figure(figsize=(7, 7))
plt.contourf(X, Y, circle_mask.astype(float), levels=[-0.5, 0.5, 1.5], alpha=0.35)
plt.plot(x, x, linestyle='--')
plt.scatter([partial_best[0]], [partial_best[1]], marker='x', s=120, label='partial stack')
plt.scatter([full_best[0]], [full_best[1]], marker='o', s=70, label='full stack')
plt.text(partial_best[0] + 0.06, partial_best[1] - 0.10, 'wrong')
plt.text(full_best[0] + 0.06, full_best[1] + 0.06, 'true')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Stable-but-wrong vs true branch')
plt.axis('equal')
plt.legend()
plt.show()

## Interpretation

- A solver can converge cleanly under a partial constraint stack.
- That does **not** guarantee the correct branch.
- Additional independent structure can change branch selection.

This is the toy version of:

> **stable solution ≠ identifiable true solution**
